# Level 7 — Maps: cobwebs, zero-change curves and stable manifolds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/worked-examples/07_maps-cobwebs-and-stable-manifolds.ipynb)

XPPAUT 8.4 (Bard Ermentrout's source of 25 November 2026) added three tools for
maps $x_{n+1} = F(x_n)$, and pyauto-xpp gives each of them to Python:

| | XPP 8.4 | pyauto-xpp | what it shows |
|---|---|---|---|
| [40](#cobweb) | Viewaxes > Cobweb (1D maps) | `pyxpp.plot.cobweb(m)` | the orbit as a staircase between the graph of $f$ and the diagonal |
| [41](#zero-change) | Nullcline, for a map | `m.nullclines()` | the zero-change curves $f(x,y) = x$ and $g(x,y) = y$; they cross at the fixed points |
| [42](#stable-manifold) | Sing pts on a saddle > draw invariant sets | `m.stable_manifold()` | the stable manifold of a saddle, computed without the inverse map |

**How to use this notebook.** Run the cells from the top (the first code cell
sets up helpers used by every section). Each section writes its model file to
`odes/<section>.ode` and its figures to `figures/<section>/`, and ends by
printing its checks. The last cell counts the checks of the whole notebook.

*The logistic and Ikeda-type models are Bard Ermentrout's examples for XPPAUT
8.4. Every number below is computed by XPPAUT through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.5.0

### Setup

`section(name)` starts a section: it sets `FIG` (the figure folder) and `ODE`
(the model file) and opens an empty list of checks. `check(what, value,
expected, tol)` compares a computed number with the expected one and records
the result; `checks_done()` prints the section's score.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pyxpp.builder import ModelBuilder

HERE = Path.cwd()
plt.rcParams.update({"figure.dpi": 100, "font.size": 11, "axes.titlesize": 12,
                     "legend.fontsize": 9, "savefig.dpi": 130})
results = {}


def section(name):
    """Start a section: its figure folder FIG, its model file ODE and an empty list of checks."""
    global NAME, FIG, ODE
    NAME = name
    FIG = HERE / "figures" / name
    FIG.mkdir(parents=True, exist_ok=True)
    ODE = HERE / "odes" / f"{name}.ode"
    ODE.parent.mkdir(exist_ok=True)
    results[name] = []


def check(what, value, expected, tol):
    ok = abs(value - expected) <= tol
    results[NAME].append(ok)
    print(f"  {'ok  ' if ok else 'FAIL'} {what}: {value:.6g} (expected {expected:.6g})")


def checks_done():
    r = results[NAME]
    print(f"\n{NAME}: {sum(r)}/{len(r)} checks passed")

In [3]:
import pyxpp
from pyxpp.plot import cobweb

<a id="cobweb"></a>
## 40. Cobwebs of the logistic map

$$
x_{n+1} = r\,x_n\,(1 - x_n).
$$

A cobweb draws an orbit on the graph of the map: go up (or down) from $x_n$ on
the diagonal to the graph, which gives $x_{n+1}$, then across to the diagonal,
and repeat. Fixed points are where the graph crosses the diagonal; one is
stable when the slope there, the multiplier $f'(x^*)$, has $|f'(x^*)| < 1$.

For this map $x^* = 1 - 1/r$ with multiplier $2 - r$: stable for $1 < r < 3$.
At $r = 3$ it turns unstable in a period doubling, and the orbit settles on a
cycle of period 2,

$$
x_\pm = \frac{r + 1 \pm \sqrt{(r - 3)(r + 1)}}{2r}.
$$

In XPP: Viewaxes > Cobweb. `cobweb(m)` draws the same picture: the diagonal
(green), the graph of $f$ (red), the orbit from the initial condition (black),
and the fixed points (filled circle: stable, open square: unstable). It uses
XPP's own right-hand side, in double precision.

In [4]:
section("40_logistic-cobweb")
mb = (ModelBuilder("logistic")
      .comment("the logistic map, for cobwebbing (XPPAUT 8.4)")
      .map("x", "r*x*(1-x)")
      .parameter(r=3.7)
      .initial(x=0.2)
      .option(meth="discrete", total=200))
print(mb.render())
m = mb.load(ODE)

# the logistic map, for cobwebbing (XPPAUT 8.4)
x(t+1)=r*x*(1-x)
par r=3.7
init x=0.2
@ meth=discrete,total=200
done



### Four values of $r$

A fixed point ($r = 2.8$), period 2 ($3.2$), period 4 ($3.5$) and chaos ($3.9$).

In [5]:
fig, axs = plt.subplots(1, 4, figsize=(16, 4.4))
data = {}
for ax, r in zip(axs, (2.8, 3.2, 3.5, 3.9)):
    m.set(r=r)
    _, data[r] = cobweb(m, n=100, ax=ax, title=f"r = {r}")
fig.tight_layout()
fig.savefig(FIG / "cobwebs.png")
plt.show()

for r, d in data.items():
    print(f"r = {r}: fixed points " + ", ".join(f"x* = {x:.6f} (multiplier {mu:+.4f})" for x, mu in d["fixed"]))

r = 2.8: fixed points x* = 0.000000 (multiplier +2.8000), x* = 0.642857 (multiplier -0.8000)
r = 3.2: fixed points x* = 0.000000 (multiplier +3.2000), x* = 0.687500 (multiplier -1.2000)
r = 3.5: fixed points x* = 0.000000 (multiplier +3.5000), x* = 0.714286 (multiplier -1.5000)
r = 3.9: fixed points x* = 0.000000 (multiplier +3.9000), x* = 0.743590 (multiplier -1.9000)


/tmp/ipykernel_355516/975412692.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### The period-2 cycle at $r = 3.2$

After a transient the staircase closes into a square: the two corners on the
diagonal are $x_\pm$.

In [6]:
m.set(r=3.2)
fig, ax = plt.subplots(figsize=(5, 5))
_, d = cobweb(m, n=8, transient=200, ax=ax, title="r = 3.2 after 200 iterations")
fig.savefig(FIG / "period2.png")
plt.show()
corners = d["orbit"][-4:, 0]                      # the last points on the diagonal
xp = (3.2 + 1 + np.sqrt((3.2 - 3) * (3.2 + 1))) / (2 * 3.2)
xm = (3.2 + 1 - np.sqrt((3.2 - 3) * (3.2 + 1))) / (2 * 3.2)
print("corners", np.round(np.unique(np.round(corners, 9)), 9), " exact", xm, xp)

corners [0.51304451 0.79945549]  exact 0.5130445095326299 0.7994554904673701


/tmp/ipykernel_355516/3609201946.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
print("checks:")
for r in (2.8, 3.9):
    x, mu = data[r]["fixed"][1]
    check(f"r = {r}: fixed point 1 - 1/r (exact)", x, 1 - 1 / r, 1e-10)
    check(f"r = {r}: multiplier 2 - r (exact)", mu, 2 - r, 1e-6)
check("r = 3.2: upper point of the 2-cycle (exact)", corners.max(), xp, 1e-9)
check("r = 3.2: lower point of the 2-cycle (exact)", corners.min(), xm, 1e-9)
checks_done()

checks:
  ok   r = 2.8: fixed point 1 - 1/r (exact): 0.642857 (expected 0.642857)
  ok   r = 2.8: multiplier 2 - r (exact): -0.8 (expected -0.8)
  ok   r = 3.9: fixed point 1 - 1/r (exact): 0.74359 (expected 0.74359)
  ok   r = 3.9: multiplier 2 - r (exact): -1.9 (expected -1.9)
  ok   r = 3.2: upper point of the 2-cycle (exact): 0.799455 (expected 0.799455)
  ok   r = 3.2: lower point of the 2-cycle (exact): 0.513045 (expected 0.513045)

40_logistic-cobweb: 6/6 checks passed


<a id="zero-change"></a>
## 41. Zero-change curves of a planar map

For a flow $\dot x = f(x,y)$, $\dot y = g(x,y)$ the nullclines are $f = 0$ and
$g = 0$. For a map $(x,y) \mapsto (f(x,y),\, g(x,y))$ the analogue is where a
variable does not change, the **zero-change curves**

$$
f(x,y) - x = 0, \qquad g(x,y) - y = 0.
$$

They cross exactly at the fixed points. Since XPPAUT 8.4 (25 Nov) the Nullcline
command draws these curves when the model is a map, so `m.nullclines()` returns
them with no change in Python.

The model is Bard's Ikeda-type map,

$$
\begin{aligned}
x &\mapsto a + b\,x\cos\mu - e\,y\sin\mu,\\
y &\mapsto b\,y\cos\mu + e\,x\sin\mu,
\end{aligned}
\qquad \mu = \varphi - \frac{q}{1 + x^2 + y^2},
$$

with $a = 1$, $b = 0.9$, $e = 1$, $\varphi = 0.4$, $q = 6$.

In [8]:
section("41_ikeda-zero-change")
mb = (ModelBuilder("ikeda")
      .comment("a planar map (Ikeda type), from Bard Ermentrout (XPPAUT 8.4)")
      .parameter(a=1, b=0.9, e=1, phi=0.4, q=6)
      .fixed("mu", "phi-q/(1+x^2+y^2)")
      .map("x", "a+b*x*cos(mu)-e*y*sin(mu)")
      .map("y", "b*y*cos(mu)+e*x*sin(mu)")
      .option(meth="discrete", total=200, xp="x", yp="y", xlo=-9, xhi=2, ylo=-5, yhi=3))
print(mb.render())
m = mb.load(ODE)

nc = m.nullclines(x="x", y="y", xlo=-9, xhi=6, ylo=-5, yhi=6, mesh=200)
print({k: v.shape for k, v in nc.items()})

# a planar map (Ikeda type), from Bard Ermentrout (XPPAUT 8.4)
par a=1,b=0.9,e=1,phi=0.4,q=6
mu=phi-q/(1+x^2+y^2)
x(t+1)=a+b*x*cos(mu)-e*y*sin(mu)
y(t+1)=b*y*cos(mu)+e*x*sin(mu)
@ meth=discrete,total=200,xp=x,yp=y,xlo=-9,xhi=2,ylo=-5,yhi=3
done



{'X': (1352, 2), 'Y': (1629, 2)}


### The fixed points

Newton's method (`m.equilibrium`, XPP's Sing pts) from the three places where
the curves cross. A fixed point of a map is stable when all its multipliers
satisfy $|\mu| < 1$, and a saddle when one is inside and one outside the unit
circle.

In [9]:
fps = [m.equilibrium(g) for g in ({"x": 1.08, "y": -2.41}, {"x": 0.51, "y": 0.27}, {"x": 2.79, "y": 4.12})]
for e in fps:
    print(f"({e.x['X']:.6f}, {e.x['Y']:.6f})   |multipliers| = {np.round(np.abs(e.eigenvalues), 4)}")

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(*nc["X"].T, color="tab:red", lw=1, label="x does not change")
ax.plot(*nc["Y"].T, color="tab:green", lw=1, label="y does not change")
for e in fps:
    mus = np.abs(e.eigenvalues)
    stable = mus.max() < 1
    ax.plot(e.x["X"], e.x["Y"], "o" if stable else "s", ms=7,
            mfc="k" if stable else "white", mec="k")
tr = m.run()
ax.plot(tr["X"], tr["Y"], ".", color="0.4", ms=3, label="an orbit (ends at the focus)")
ax.set(xlim=(-9, 6), ylim=(-5, 6), xlabel="x", ylabel="y", title="Zero-change curves and fixed points")
ax.legend(loc="lower right", fontsize=8)
fig.savefig(FIG / "zero_change.png")
plt.show()

(1.083319, -2.407963)   |multipliers| = [1.6402 0.4585]
(0.510744, 0.273108)   |multipliers| = [0.3058 2.394 ]
(2.788310, 4.122981)   |multipliers| = [0.9106 0.9106]


/tmp/ipykernel_355516/3055836399.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
def F(x, y, a=1, b=0.9, e=1, phi=0.4, q=6):
    mu = phi - q / (1 + x * x + y * y)
    return a + b * x * np.cos(mu) - e * y * np.sin(mu), b * y * np.cos(mu) + e * x * np.sin(mu)

print("checks:")
for e in fps:
    fx, fy = F(e.x["X"], e.x["Y"])
    check(f"F(p) = p at ({e.x['X']:.3f}, {e.x['Y']:.3f}) (to Sing pts' Newton tolerance)",
          np.hypot(fx - e.x["X"], fy - e.x["Y"]), 0, 1e-6)
for k, other in (("X", "Y"), ("Y", "X")):
    pts = nc[k][np.isfinite(nc[k][:, 0])][::25]
    change = [m.rhs({"x": u, "y": v})[k] - {"X": u, "Y": v}[k] for u, v in pts]
    check(f"{k.lower()} does not change on its curve (grid accuracy)", np.max(np.abs(change)), 0, 0.02)
types = sorted("saddle" if (np.abs(e.eigenvalues) > 1).sum() == 1 else "stable" for e in fps)
check("two saddles and one stable fixed point", float(types == ["saddle", "saddle", "stable"]), 1, 0)
checks_done()

checks:
  ok   F(p) = p at (1.083, -2.408) (to Sing pts' Newton tolerance): 2.1777e-09 (expected 0)
  ok   F(p) = p at (0.511, 0.273) (to Sing pts' Newton tolerance): 2.35486e-08 (expected 0)
  ok   F(p) = p at (2.788, 4.123) (to Sing pts' Newton tolerance): 4.38814e-10 (expected 0)


  ok   x does not change on its curve (grid accuracy): 0.0009644 (expected 0)


  ok   y does not change on its curve (grid accuracy): 0.00268922 (expected 0)
  ok   two saddles and one stable fixed point: 1 (expected 1)

41_ikeda-zero-change: 6/6 checks passed


<a id="stable-manifold"></a>
## 42. The stable manifold of a saddle, without the inverse

The stable manifold $W^s(p)$ of a saddle $p$ is the set of points whose orbits
end at $p$. For a flow one integrates backwards from points next to $p$ along
the stable eigenvector. A map cannot always be run backwards: here the inverse
has no formula.

XPPAUT 8.4 computes $W^s$ with the method of England, Krauskopf and Osinga
(*SIAM J. Appl. Dyn. Syst.* 3 (2004) 161–190), which needs only $F$. The
manifold is grown point by point in order of arclength. Each new point $z$ lies
on a small circle around the last one, chosen so that $F(z)$ falls on the part
of the manifold already found: $W^s$ is invariant, so $F(W^s) \subset W^s$. The
step size adapts to the curvature.

`m.stable_manifold(near=...)` polishes the saddle by Newton's method and
returns both branches as arrays. Its settings are XPP's (Numerics > stabLe
manifold, or `@ MFLEN=...` in the .ode file):
- `arclength` (100) and `init` (1e-3);
- the bounds on the turning angle (`alpha`), the deviation (`dalpha`) and the step (`delta`);
- `box`: by default the view enlarged by half its size on each side, as in XPP.

In [11]:
section("42_ikeda-stable-manifold")
m.view("x", "y", -9, 2, -5, 3)
ws = m.stable_manifold(near={"x": 1.08, "y": -2.41})
print("saddle", ws["saddle"], " stable multiplier", ws["multiplier"])
for i, (b, why, L) in enumerate(zip(ws["branches"], ws["why"], ws["arclength"])):
    print(f"branch {i}: {len(b)} points, arclength {L:.2f}, {why}")

fig, ax = plt.subplots(figsize=(7, 5.5))
for b, c in zip(ws["branches"], ("tab:blue", "tab:purple")):
    ax.plot(b[:, 0], b[:, 1], color=c, lw=0.7)
ax.plot(*nc["X"].T, color="tab:red", lw=0.6, alpha=0.5)
ax.plot(*nc["Y"].T, color="tab:green", lw=0.6, alpha=0.5)
ax.plot(ws["saddle"]["X"], ws["saddle"]["Y"], "s", mfc="white", mec="k", ms=7)
ax.set(xlim=(-9, 2), ylim=(-5, 3), xlabel="x", ylabel="y",
       title="Stable manifold of the saddle (blue, purple), zero-change curves")
fig.savefig(FIG / "stable_manifold.png")
plt.show()

saddle {'X': 1.0833188740400526, 'Y': -2.407963483400624}  stable multiplier 0.4582248070021441
branch 0: 5333 points, arclength 17.88, left the region
branch 1: 50580 points, arclength 100.00, reached the arclength


/tmp/ipykernel_355516/2196875367.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Checks

* The manifold is invariant: for points $P$ on it, $F(P)$ lies on it again (to
  the accuracy of the polyline).
* The saddle and its stable multiplier agree with Sing pts (`m.equilibrium`).
  Sing pts takes its Jacobian from forward differences, good to about $10^{-3}$;
  the manifold code first polishes the saddle by Newton's method with central
  differences, so its multiplier is checked more strictly against the exact
  Jacobian of $F$ (central differences in numpy).
* An orbit started on the manifold gets closer and closer to the saddle.

In [12]:
def dist_to_polyline(px, py, B):
    A, D = B[:-1], np.diff(B, axis=0)
    L2 = (D ** 2).sum(1) + 1e-300
    t = np.clip(((px[:, None] - A[:, 0]) * D[:, 0] + (py[:, None] - A[:, 1]) * D[:, 1]) / L2, 0, 1)
    return np.min(np.hypot(A[:, 0] + t * D[:, 0] - px[:, None], A[:, 1] + t * D[:, 1] - py[:, None]), axis=1)

worst = 0
for b in ws["branches"]:
    P = b[np.linspace(1, len(b) - 1, 300).astype(int)]
    fx, fy = F(P[:, 0], P[:, 1])
    d = np.minimum(dist_to_polyline(fx, fy, ws["branches"][0]), dist_to_polyline(fx, fy, ws["branches"][1]))
    worst = max(worst, d.max())

eq = fps[0]
mu_s = min(eq.eigenvalues, key=abs).real
sx, sy, h = ws["saddle"]["X"], ws["saddle"]["Y"], 1e-6
J = np.column_stack([(np.array(F(sx + h, sy)) - F(sx - h, sy)) / (2 * h),
                     (np.array(F(sx, sy + h)) - F(sx, sy - h)) / (2 * h)])
mu_exact = min(np.linalg.eigvals(J), key=abs).real
z = ws["branches"][0][len(ws["branches"][0]) // 50]          # a point on the manifold near the saddle
dists = []
for k in range(12):
    dists.append(np.hypot(z[0] - ws["saddle"]["X"], z[1] - ws["saddle"]["Y"]))
    z = F(*z)
print("distance to the saddle along an orbit on W^s:", np.array2string(np.array(dists), precision=2))

print("checks:")
check("F(W^s) lies on W^s (largest distance)", worst, 0, 1e-5)
check("saddle x agrees with Sing pts", ws["saddle"]["X"], eq.x["X"], 1e-6)
check("saddle y agrees with Sing pts", ws["saddle"]["Y"], eq.x["Y"], 1e-6)
check("stable multiplier = that of the exact Jacobian", ws["multiplier"], mu_exact, 1e-7)
check("stable multiplier agrees with Sing pts (forward differences)", ws["multiplier"], mu_s, 1e-3)
check("orbit on W^s: distance shrinks by the multiplier each step", dists[-1] / dists[-2], ws["multiplier"], 0.01)
checks_done()

distance to the saddle along an orbit on W^s: [2.74e-01 1.28e-01 5.92e-02 2.72e-02 1.25e-02 5.73e-03 2.63e-03 1.20e-03
 5.52e-04 2.53e-04 1.15e-04 5.22e-05]
checks:
  ok   F(W^s) lies on W^s (largest distance): 1.0359e-06 (expected 0)
  ok   saddle x agrees with Sing pts: 1.08332 (expected 1.08332)
  ok   saddle y agrees with Sing pts: -2.40796 (expected -2.40796)
  ok   stable multiplier = that of the exact Jacobian: 0.458225 (expected 0.458225)
  ok   stable multiplier agrees with Sing pts (forward differences): 0.458225 (expected 0.458539)
  ok   orbit on W^s: distance shrinks by the multiplier each step: 0.452412 (expected 0.458225)

42_ikeda-stable-manifold: 6/6 checks passed


## Summary

In [13]:
for name, r in results.items():
    print(f"{name:40s} {sum(r):3d}/{len(r)}")
n_ok = sum(sum(r) for r in results.values())
n = sum(len(r) for r in results.values())
print(f"\n{n_ok}/{n} checks passed")

40_logistic-cobweb                         6/6
41_ikeda-zero-change                       6/6
42_ikeda-stable-manifold                   6/6

18/18 checks passed
